# Lab 11: Grid Localization using Bayes Filter (Real Robot)

### <span style="color:rgb(0,150,0)">It is recommended that you close any heavy-duty applications running on your system while working on this lab.</span>

### <span style="color:rgb(0,150,0)">The notebook only provides skeleton code for you to integrate the Localization class with the Real Robot.</span>

<hr>

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import asyncio
import pathlib
import os
import numpy as np

import traceback
from notebook_utils import *
from Traj import *
from utils import load_config_params
from localization_extras import Localization

# Make the project's main_python BLE client importable.
_BLE_ROOT = pathlib.Path(os.getcwd()).resolve().parent.parent / "ble_robot_1.4"
if str(_BLE_ROOT) not in sys.path:
    sys.path.insert(0, str(_BLE_ROOT))

from main_python.ble.connection import BLEConnection
from main_python.commands import MapStart, SendMapData, MapStatus

# Setup Logger
LOG = get_logger('demo_notebook.log')
LOG.propagate = False

# Init GUI and Commander
gui = GET_GUI()
cmdr = gui.launcher.commander


In [ ]:
# Start the plotter
START_PLOTTER()

# The RealRobot class
Define the RealRobot class in the code cell below, based on the documentation and your real robot communication protocol. <br>
This class is used by the **Localization** class to communicate with the real robot. <br>
More specifically, the **Localization** class utilizes the **RealRobot's** member function **perform_observation_loop()** to get the 18 sensor readings and store them in its member variable **obs_range_data**, which is then utilized in the update step.

In [ ]:
class RealRobot():
    """A class to interact with the real robot using the async main_python BLE client."""

    def __init__(self, commander, ble):
        self.world_config = os.path.join(
            str(pathlib.Path(os.getcwd()).parent), "config", "world.yaml"
        )
        self.config_params = load_config_params(self.world_config)
        self.cmdr = commander
        self.ble = ble  # BLEConnection (already entered)

    def get_pose(self):
        raise NotImplementedError("get_pose is not implemented")

    async def perform_observation_loop(self, rot_vel=120):
        """Drive the firmware mapping subsystem through an 18-step 360 sweep
        and return the per-bucket ToF means as an (18, 1) numpy column array
        in meters. Bearings are not used by the localization module, so an
        empty array is returned.
        """
        await self.ble.execute(MapStart(num_steps=18))
        while await self.ble.execute(MapStatus()):
            await asyncio.sleep(0.5)

        buckets = await self.ble.execute(SendMapData())
        buckets.sort(key=lambda b: b.index)
        ranges_m = np.array([b.distance / 1000.0 for b in buckets])
        return ranges_m[np.newaxis].T, np.array([])


In [ ]:
# Open the async BLE connection to the Artemis. The connection stays
# open for the lifetime of the notebook session.
ble = BLEConnection()
await ble.__aenter__()


In [ ]:
# Initialize RealRobot with a Commander object to communicate with the plotter process
# and the ArtemisBLEController object to communicate with the real robot
robot = RealRobot(cmdr, ble)

# Initialize mapper
# Requires a VirtualRobot object as a parameter
mapper = Mapper(robot)

# Initialize your BaseLocalization object
# Requires a RealRobot object and a Mapper object as parameters
loc = Localization(robot, mapper)

## Plot Map
cmdr.plot_map()

# Run an update step of the Bayes Filter

In [ ]:
# Reset Plots
cmdr.reset_plotter()

# Init Uniform Belief
loc.init_grid_beliefs()

# Get Observation Data by executing a 360 degree rotation motion
await loc.get_observation_data()

# Run Update Step
loc.update_step()
loc.plot_update_step_data(plot_data=True)

# Plot Odom and GT
# current_odom, current_gt = robot.get_pose()
# cmdr.plot_gt(current_gt[0], current_gt[1])
# cmdr.plot_odom(current_odom[0], current_odom[1])
